# 1. Data exploration

Paris PM2.5 daily means (OpenAQ). Owner: Prem — Phase 1.

In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_root))
import os
os.chdir(project_root)

from src.eda import adf_table, count_missing, impute_linear, rolling_mean_std, suggest_sarima_orders
from src.load_data import load_daily_series
from src.plots import plot_acf_pacf, plot_rolling, plot_series, save_current_figure, save_figure

DATA_PATH = project_root / "data/processed/pm25_paris_daily.csv"
(project_root / "reports/figures").mkdir(parents=True, exist_ok=True)

raw = load_daily_series(DATA_PATH)
raw.head()

date
2023-01-01    10.004167
2023-01-02     6.608333
2023-01-03     7.954167
2023-01-04     4.141667
2023-01-05     4.937500
Freq: D, Name: pm25, dtype: float64

## Full series, zoomed month, and rolling mean/std

In [2]:
ax = plot_series(raw.dropna(), title="Paris PM2.5 — full series")
save_figure(ax, "01_full_series.png")
plt.show()

/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_18579/2040132322.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
zoom_end = raw.dropna().index.max()
zoom_start = zoom_end - pd.Timedelta(days=30)
zoom = raw.loc[zoom_start:zoom_end].dropna()
ax = plot_series(zoom, title=f"Paris PM2.5 — zoomed month ({zoom_start.date()} to {zoom_end.date()})")
save_figure(ax, "01_zoomed_month.png")
plt.show()

/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_18579/4148555015.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
roll_mean, roll_std = rolling_mean_std(raw.dropna(), window=7)
ax = plot_rolling(raw.dropna(), roll_mean, roll_std, title="7-day rolling mean and std")
save_figure(ax, "01_rolling_mean_std.png")
plt.show()

/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_18579/3275540482.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Missing timestamps and imputation

**Choice:** linear interpolation on the daily calendar (`impute_linear`), with edge `ffill`/`bfill` only if the first/last days are missing.

**Why not forward-fill:** ffill creates flat segments that artificially inflate ACF at lag 1. Short sensor outages in PM2.5 are better filled by interpolating between neighbouring days.

In [5]:
missing = count_missing(raw)
print(
    f"expected_days={missing.expected_days}\n"
    f"observed_days={missing.observed_days}\n"
    f"missing_timestamps={missing.missing_timestamps}\n"
    f"missing_values_in_file={missing.missing_values}"
)
if len(missing.missing_dates):
    print("first missing dates:", list(missing.missing_dates[:10]))

series = impute_linear(raw)
assert series.isna().sum() == 0
series.describe()

expected_days=1034
observed_days=1014
missing_timestamps=20
missing_values_in_file=20
first missing dates: [Timestamp('2023-01-10 00:00:00'), Timestamp('2023-04-05 00:00:00'), Timestamp('2023-06-13 00:00:00'), Timestamp('2023-07-25 00:00:00'), Timestamp('2023-09-28 00:00:00'), Timestamp('2023-10-13 00:00:00'), Timestamp('2023-11-02 00:00:00'), Timestamp('2023-12-03 00:00:00'), Timestamp('2023-12-18 00:00:00'), Timestamp('2024-01-13 00:00:00')]


count    1034.000000
mean       10.794294
std         7.095248
min         2.479167
25%         6.081771
50%         8.618750
75%        13.177083
max        55.162500
Name: pm25, dtype: float64

## ADF stationarity tests

In [6]:
adf = adf_table(series)
display(adf)
raw_stationary = bool(adf.loc[adf["series"] == "raw", "stationary_5pct"].iloc[0])
print(
    "Raw series is",
    "stationary at 5%. Use d=0" if raw_stationary else "NOT stationary at 5%. Prefer d=1",
    "for SARIMA.",
)

/Users/prem/tsa-group5-air-quality/src/eda.py:56: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pvalue, used_lag, nobs, *_ = adfuller(clean, autolag="AIC")
/Users/prem/tsa-group5-air-quality/src/eda.py:56: FutureWarning: adfuller currently returns a plain tuple whose length depends on the store and autolag arguments. In release 0.16 or after July 2027, whichever is later, the default behavior will switch to always returning an ADFullerResult. Set result_object=True to switch now, or result_object=False to keep the current behavior and silence this warning.
  stat, pvalue, used_lag, nobs, *_ = adfuller(clean, autolag="AIC")


,series,adf_stat,pvalue,used_lag,nobs,stationary_5pct
0,raw,-4.419287,2.743525e-04,14,1019,True
1,diff_1,-11.654238,2.003213e-21,18,1014,True


Raw series is stationary at 5%. Use d=0 for SARIMA.


## ACF/PACF and suggested orders

Inspect lag-1 behaviour for `(p, q)` and lags 7 / 14 for weekly seasonal `(P, Q, s=7)`.

In [7]:
plot_acf_pacf(series, lags=40)
save_current_figure("01_acf_pacf.png")
plt.show()

prefer_diff = not raw_stationary
orders = suggest_sarima_orders(seasonal_period=7, prefer_diff=prefer_diff)
print("Suggested order:", orders["order"])
print("Suggested seasonal_order:", orders["seasonal_order"])
print("Hand these to Yahya (also in docs/sarima_orders_for_yahya.md).")

Suggested order: (1, 0, 1)
Suggested seasonal_order: (1, 1, 1, 7)
Hand these to Yahya (also in docs/sarima_orders_for_yahya.md).


/var/folders/8n/mwqyfv8j437btzxfh5c852n00000gn/T/ipykernel_18579/2444714836.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
